In [ ]:
from sts.data.ticker import Ticker
import plotly.io as pio
import pandas as pd
import numpy as np
from sts.quant.candle import Candle
from sts.quant.consolidate.consolidate import get_consolidate_score_mm_multi_horizon
import datetime

ticker = Ticker("us/etf.yml")
td = datetime.today()
etf_symbols = ticker.top_trading_symbols(td, top_n=100)
sdate = td - datetime.timedelta(days=200)
edate = td
weekly_consolidates_etf = []
for etf_sym in etf_symbols:
    df = ticker.history(etf_sym, sdate, edate).set_index("ts")
    df.index = pd.to_datetime(df.index, utc=True)
    score_rets = get_consolidate_score_mm_multi_horizon(df, window=20, buffer_threshold=0.2, sample_rules=["1D", "1W"])
    daily_score = score_rets["DailyScore"].rolling(window=5).mean().iloc[-1]
    weekly_score = score_rets["WeeklyScore"].rolling(window=5).mean().iloc[-1]
    if weekly_score > 0.5:
        weekly_consolidates_etf.append({"sym": etf_sym, "weekly_score": weekly_score, "daily_score": "daily_score"})
if len(weekly_consolidates_etf) > 0:
    weekly_consolidates_etf = pd.DataFrame(weekly_consolidates_etf)